# Schwartz-Smith As GBM Check

This notebook forces the Schwartz-Smith model to behave like a simple GBM/lognormal model, then simulates both side by side.

The full Schwartz-Smith model is:

$$\log P_t = \chi_t + \xi_t$$

$$d\chi_t = -\kappa \chi_t dt + \sigma_\chi dW_\chi$$

$$d\xi_t = \mu_\xi dt + \sigma_\xi dW_\xi$$

To make it behave like GBM, set the short factor volatility almost to zero:

$$\sigma_\chi \approx 0$$

Then:

$$\log P_t \approx \xi_t$$

and price follows a lognormal process.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.processes.schwartz_smith import SSParams, simulate_paths
from src.config import SCHWARTZ_SMITH

RAW_DIR = PROJECT_ROOT / "data" / "raw"

plt.style.use("seaborn-v0_8-whitegrid")
print(f"Project root: {PROJECT_ROOT}")

## 1. Choose GBM Inputs

This estimates GBM drift from daily average day-ahead prices if the raw Elexon parquet exists, while using a fixed 25% annual volatility. Otherwise it falls back to simple assumptions.

In [ ]:
def estimate_gbm_inputs(raw_dir):
    da_path = raw_dir / "elexon_da_prices.parquet"
    if not da_path.exists():
        return None

    df = pd.read_parquet(da_path)
    df["settlement_date"] = pd.to_datetime(df["settlement_date"])
    daily = (df.groupby("settlement_date")["price_gbp_mwh"]
               .mean()
               .sort_index())
    daily = daily[daily > 0]
    log_ret = np.log(daily).diff().dropna()

    if len(log_ret) < 10:
        return None

    dt = 1 / 365.25
    sigma = 0.25  # fixed 25% annual volatility
    log_drift = float(log_ret.mean() / dt)
    mu = log_drift + 0.5 * sigma**2

    return {
        "S0": float(daily.iloc[-1]),
        "mu": mu,
        "sigma": sigma,
        "source": str(da_path),
        "n_returns": int(len(log_ret)),
    }

gbm = estimate_gbm_inputs(RAW_DIR)

if gbm is None:
    gbm = {
        "S0": float(SCHWARTZ_SMITH["forward_anchor_gbp_mwh"]),
        "mu": 0.00,
        "sigma": 0.25,
        "source": "fallback assumptions",
        "n_returns": 0,
    }

print("GBM inputs")
print(f"  S0:       £{gbm['S0']:.2f}/MWh")
print(f"  mu:       {gbm['mu']:.4f} per year")
print(f"  sigma:    {gbm['sigma']:.4f} per sqrt(year)")
print(f"  source:   {gbm['source']}")
print(f"  n returns: {gbm['n_returns']}")

## 2. Convert Schwartz-Smith Into GBM

The direct GBM process has:

$$dS_t = \mu S_t dt + \sigma S_t dW_t$$

which means:

$$d\log S_t = (\mu - 0.5\sigma^2)dt + \sigma dW_t$$

The current `simulate_paths()` implementation applies `mu_xi` directly to the log process, so we set:

$$\mu_\xi = \mu - 0.5\sigma^2$$

and:

$$\sigma_\xi = \sigma$$

In [ ]:
S0 = float(gbm["S0"])
mu_price = float(gbm["mu"])
sigma_price = float(gbm["sigma"])
mu_log = mu_price - 0.5 * sigma_price**2

ss_as_gbm = SSParams(
    kappa=5.0,              # irrelevant when sigma_chi is almost zero
    mu_xi=mu_log,           # log-price drift
    sigma_chi=1e-10,        # switches off short mean-reverting factor
    sigma_xi=sigma_price,   # GBM volatility
    rho=0.0,                # irrelevant when sigma_chi is almost zero
)

pd.DataFrame([{
    "S0": S0,
    "gbm_mu_price": mu_price,
    "gbm_sigma": sigma_price,
    "ss_kappa": ss_as_gbm.kappa,
    "ss_mu_xi_log_drift": ss_as_gbm.mu_xi,
    "ss_sigma_chi": ss_as_gbm.sigma_chi,
    "ss_sigma_xi": ss_as_gbm.sigma_xi,
    "ss_rho": ss_as_gbm.rho,
}]).T.rename(columns={0: "value"})

## 3. Simulate Direct GBM And SS-As-GBM

In [ ]:
def simulate_direct_gbm(S0, mu, sigma, n_paths, n_steps, dt, seed=42):
    rng = np.random.default_rng(seed)
    z = rng.standard_normal((n_paths, n_steps))
    log_increments = (mu - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * z

    log_paths = np.empty((n_paths, n_steps + 1), dtype=float)
    log_paths[:, 0] = np.log(S0)
    log_paths[:, 1:] = np.log(S0) + np.cumsum(log_increments, axis=1)
    return np.exp(log_paths), log_paths

N_PATHS = 5000
N_STEPS = 365
DT = 1 / 365.25
SEED = 123

price_gbm, log_gbm = simulate_direct_gbm(
    S0=S0,
    mu=mu_price,
    sigma=sigma_price,
    n_paths=N_PATHS,
    n_steps=N_STEPS,
    dt=DT,
    seed=SEED,
)

# Simulate SS factors, then anchor xi_0 at log(S0).
chi_ss, xi_ss = simulate_paths(
    ss_as_gbm,
    n_paths=N_PATHS,
    n_steps=N_STEPS,
    dt=DT,
    seed=SEED,
)
log_ss = chi_ss + xi_ss + np.log(S0)
price_ss = np.exp(np.clip(log_ss, -100, np.log(5000)))

print("Direct GBM:", price_gbm.shape, "finite:", np.isfinite(price_gbm).all())
print("SS as GBM: ", price_ss.shape, "finite:", np.isfinite(price_ss).all())

## 4. Plot Sample Paths Together

In [ ]:
days = np.arange(N_STEPS + 1)
sample_n = 12

plt.figure(figsize=(12, 6))
for i in range(sample_n):
    plt.plot(days, price_gbm[i], color="tab:blue", alpha=0.35, lw=1)
    plt.plot(days, price_ss[i], color="tab:orange", alpha=0.35, lw=1, ls="--")

plt.plot([], [], color="tab:blue", label="Direct GBM")
plt.plot([], [], color="tab:orange", ls="--", label="Schwartz-Smith parameterized as GBM")
plt.axhline(S0, color="black", lw=1, ls=":", label="S0")
plt.xlabel("days")
plt.ylabel("£/MWh")
plt.title("Sample Paths: Direct GBM vs Schwartz-Smith-As-GBM")
plt.legend()
plt.ylim(0, min(500, max(np.percentile(price_gbm, 99), np.percentile(price_ss, 99))))
plt.show()

## 5. Fan Chart Comparison

In [ ]:
def fan(paths):
    return {q: np.percentile(paths, q, axis=0) for q in [5, 25, 50, 75, 95]}

gbm_fan = fan(price_gbm)
ss_fan = fan(price_ss)

fig, ax = plt.subplots(figsize=(12, 6))
ax.fill_between(days, gbm_fan[5], gbm_fan[95], color="tab:blue", alpha=0.12)
ax.fill_between(days, gbm_fan[25], gbm_fan[75], color="tab:blue", alpha=0.22)
ax.plot(days, gbm_fan[50], color="tab:blue", lw=2, label="Direct GBM median")

ax.fill_between(days, ss_fan[5], ss_fan[95], color="tab:orange", alpha=0.12)
ax.fill_between(days, ss_fan[25], ss_fan[75], color="tab:orange", alpha=0.22)
ax.plot(days, ss_fan[50], color="tab:orange", lw=2, ls="--", label="SS-as-GBM median")

ax.axhline(S0, color="black", lw=1, ls=":", label="S0")
ax.set_xlabel("days")
ax.set_ylabel("£/MWh")
ax.set_title("Fan Chart: Direct GBM vs Schwartz-Smith-As-GBM")
ax.set_ylim(0, min(500, max(np.percentile(price_gbm, 99), np.percentile(price_ss, 99))))
ax.legend()
plt.show()

## 6. Terminal Distributions

In [ ]:
terminal_gbm = price_gbm[:, -1]
terminal_ss = price_ss[:, -1]
cap = min(500, max(np.percentile(terminal_gbm, 99), np.percentile(terminal_ss, 99)))
bins = np.linspace(0, cap, 80)

plt.figure(figsize=(10, 5))
plt.hist(terminal_gbm, bins=bins, alpha=0.50, density=True, label="Direct GBM")
plt.hist(terminal_ss, bins=bins, alpha=0.50, density=True, label="SS-as-GBM")
plt.axvline(S0, color="black", lw=1, ls=":", label="S0")
plt.xlabel("terminal price after 1 year (£/MWh)")
plt.ylabel("density")
plt.title("Terminal Price Distribution")
plt.legend()
plt.show()

## 7. Summary Statistics

In [ ]:
def summarize(name, prices, logs):
    terminal = prices[:, -1]
    log_ret = logs[:, -1] - logs[:, 0]
    return {
        "model": name,
        "mean_terminal": terminal.mean(),
        "median_terminal": np.median(terminal),
        "p05_terminal": np.percentile(terminal, 5),
        "p95_terminal": np.percentile(terminal, 95),
        "mean_log_return": log_ret.mean(),
        "std_log_return": log_ret.std(ddof=1),
    }

summary = pd.DataFrame([
    summarize("Direct GBM", price_gbm, log_gbm),
    summarize("SS-as-GBM", price_ss, log_ss),
])

display(summary.round(4))

diff = summary.set_index("model").loc["SS-as-GBM"] - summary.set_index("model").loc["Direct GBM"]
print("SS-as-GBM minus Direct GBM:")
display(diff.to_frame("difference").round(6))

## 8. Interpretation

The two simulations should be close in distribution, but they will not be path-by-path identical because `simulate_paths()` draws two random factors and uses an exact covariance matrix, while the direct GBM uses a single normal draw.

For practical use:

- `sigma_chi = 0` turns off the short-term mean-reverting factor.
- `sigma_xi = GBM sigma` keeps the long-term lognormal volatility.
- `mu_xi = GBM mu - 0.5 * sigma^2` because the SS simulator evolves log-price directly.
- `xi_0 = log(S0)` anchors the initial price level.